# LlamaIndex Components

In [20]:
from huggingface_hub import login

from datasets import load_dataset
from pathlib import Path
import nest_asyncio

import chromadb

from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.evaluation import FaithfulnessEvaluator

In [3]:
login()

We can start by downloading a dataset of persons that will be invited to a party we are organizing:

In [4]:
dataset = load_dataset(path='dvilasuero/finepersonas-v0.1-tiny', split='train')

DATA_DIR = Path('../data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
for i, persona in enumerate(dataset):
    with open(DATA_DIR / f"persona_{i}.txt", 'w') as f:
        f.write(persona['persona'])

In [5]:
reader = SimpleDirectoryReader(input_dir=DATA_DIR)
documents = reader.load_data()
len(documents)

5000

In [6]:
documents[0]

Document(id_='2c138966-1f55-4b50-9433-4adbac1ee280', embedding=None, metadata={'file_path': '/Users/stepit/Repositories/projects/hugging-face-agents-course/notebooks/../data/persona_0.txt', 'file_name': 'persona_0.txt', 'file_type': 'text/plain', 'file_size': 132, 'creation_date': '2026-10-14', 'last_modified_date': '2026-10-14'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='A local art historian and museum professional interested in 19th-century American art and the local cultural heritage of Cincinnati.', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n

Now we want to create embeddings for these documents:

In [7]:
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
    ],
)

nodes = await pipeline.arun(documents=documents[:10])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
nodes[0]

TextNode(id_='cdf74ee0-207a-4cff-9f63-7ba494ba27bb', embedding=[-0.04365457966923714, 0.022888336330652237, 0.02666684426367283, -0.00254265614785254, -0.021060993894934654, -0.00664172088727355, -0.013476498425006866, -0.013759615831077099, -0.049493007361888885, -0.044402219355106354, 0.01910579949617386, -0.03571029379963875, -0.03020455688238144, 0.028886893764138222, -0.0023785773664712906, -0.02876509539783001, -0.004296960309147835, 0.09704290330410004, 0.027140513062477112, 0.021493451669812202, -0.04129979386925697, -0.056279152631759644, 0.05399920418858528, -0.031108735129237175, -0.03533926233649254, 0.0190778449177742, 0.05158831924200058, -0.013684078119695187, 0.011644221842288971, -0.1212548241019249, -0.0368979275226593, 0.0355430468916893, -0.03168787062168121, 0.021260982379317284, 0.08522322028875351, 0.038521070033311844, 0.01856105960905552, 0.04456745460629463, -0.0015157986199483275, -0.0012284950353205204, 0.016137875616550446, 0.02110428363084793, -0.023116834

We now have to add these nodes to a vector database. We can directly add this phase in the ingestion pipeline. To do so, we will use an open-source search infrastructure for AI called [Chroma](https://www.trychroma.com/). With it, we can:
- Store embeddings and metadata
- search with dense or sparse vectors
- filter by metadata
- and more...

Notice that we use only 10 documents to avoid crashing the notebook for out of memory.

In [12]:
embedding_model = HuggingFaceEmbedding(
    model_name='BAAI/bge-small-en-v1.5',
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
db = chromadb.PersistentClient(path='../alfred_chroma_db')
chroma_collection = db.get_or_create_collection(name='alfred')
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        embedding_model,
    ],
    vector_store=vector_store,
)

nodes = await pipeline.arun(documents=documents[:10])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [10]:
len(nodes)

10

Now we create a vector store index for the vector store just created. Indexes are required to effectively search through generated nodes.

In [14]:
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embedding_model
)

Let's see now how to query the documents in the vector db.

In [17]:
nest_asyncio.apply()

llm = HuggingFaceInferenceAPI(
    model_name='meta-llama/Llama-3.3-70B-Instruct',
    temperature=0.7,
    max_tokens=100,
    provider='auto',
)

query_engine = index.as_query_engine(
    llm=llm,
    response_mode='tree_summarize',
)

In [18]:
resp = query_engine.query(
    "Respond using a persona that describes author and travel experiences?"
)

In [19]:
resp

Response(response="As an anthropologist, I've had the privilege of immersing myself in the rich cultural heritage of Cyprus, where I've spent a significant amount of time researching and living among the locals. My travels have taken me to the sun-kissed beaches, ancient ruins, and vibrant cities of this enchanting island, allowing me to gain a deep understanding of its people, customs, and way of life. From attending traditional festivals to sampling local cuisine, I've been fortunate enough to experience the warm hospitality of the Cypriot people firsthand. My adventures have not only broadened my knowledge but also instilled in me a profound appreciation for the intricate complexities of Cypriot culture and history.", source_nodes=[NodeWithScore(node=TextNode(id_='d6ca5327-3d6b-4816-afe6-4e5a88fe0bb7', embedding=None, metadata={'file_path': '/Users/stepit/Repositories/projects/hugging-face-agents-course/notebooks/../data/persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'te

LlamaIndex provides out-of-the-box a tool to assess the quality of responses.

In [21]:
evaluator = FaithfulnessEvaluator(llm=llm)
eval_result = evaluator.evaluate_response(response=resp)
eval_result.passing

True